# Environnement Image

Sessions image (Z-Image / Krea) sans network volume. L'env et les modeles sont restaures depuis HF (`rafatribe/Personnal_Loras`).

**Routine**
1. Cellule **1** : boot (env + modeles), quelques minutes.
2. Cellule **3** : lancement de ComfyUI.
3. Travailler. Outputs, inputs et workflows sont sur le Volume Disk : ils survivent a un Stop.
4. Fin de journee : cellule **5** pour zipper les outputs, telechargement, puis Terminate.

Maintenance (rarement) : cellules 6 (mise a jour), 7 (snapshot), 8 (liste des modeles).


## 1. Boot de la session (env + modeles)
Choisis les profils dans l'encadre. Le boot verifie la place avant de telecharger. Un modele absent reste telechargeable via le Model Manager.


In [ ]:
%%bash
# ======================= PROFILS DE MODELES =======================
# Un ou plusieurs, separes par des virgules, sans espace.
#   krea2        Krea 2 + LoRAs Krea                                    ~37 Go
#   zimage       Z-Image Turbo + Base + NSFW v2, Juggernaut, RealDream,
#                DarkBeast, ControlNet, tes LoRAs Z-Image               ~97 Go
#   qwen         Qwen Image Edit 2511 fp8 + Lightning                   ~31 Go
#   flux         Flux 2 Klein 9B (fp8, base) + Flux.1 dev + LoRAs       ~70 Go
#   sdxl         SDXL (base, JibMix, Lustify, SUPIR) + SD 1.5           ~39 Go
#   anima        Anima preview                                          ~6 Go
#   tout         tous les modeles de l'ancien volume image              ~77 Go
#   (le profil "base" : detection visage, upscalers, 2 Go, est toujours ajoute)
# Exemples : "zimage"   "zimage,qwen"   "sdxl,anima"
# Volume Disk : additionne les tailles + ~10 Go de marge.
export PROFILS="krea2"
# ===================================================================
export SESSION_ENV="image"

if [ -f /workspace/ComfyUI/main.py ] && [ ! -L /workspace/ComfyUI ] && [ ! -f /workspace/.env_restored ]; then
  echo "Pod d'origine detecte (ComfyUI deja present dans /workspace) : boot inutile."
  exit 0
fi
curl -sL -H "Authorization: Bearer $HF_TOKEN" https://huggingface.co/rafatribe/Personnal_Loras/resolve/main/session/boot.sh | bash


## 2. Suivi du telechargement des modeles
A relancer autant de fois que tu veux pendant le boot.


In [ ]:
%%bash
LOG=/local/fetch_models.log
[ -f "$LOG" ] || LOG=/workspace/fetch_models.log
echo "$(grep -c '^\[' "$LOG" 2>/dev/null || echo 0) fichiers traites"
tail -n 15 "$LOG" 2>/dev/null
df -h / | tail -1 | awk '{print "Container Disk : " $4 " libres sur " $2}'

## 3. Lancement de ComfyUI
Verifie le driver, applique les hotfixes (NAG, ImageMagick 6, llama-cpp), puis lance ComfyUI. A lancer apres la fin du boot (cellule 1).


In [ ]:
# ==================================================================================
# ÉTAPE -1 — CHECK DRIVER (nvvfx / Nvidia_RTX_Nodes => driver >= 570.26)
# ==================================================================================
import subprocess

NVVFX_MIN_DRIVER = (570, 26)  # NVIDIA Video Effects SDK (Maxine VFX)

def check_nvvfx_compat():
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=driver_version,name", "--format=csv,noheader"],
            capture_output=True, text=True).stdout.strip()
        drv_str, gpu = [x.strip() for x in out.split(",")]
        p = drv_str.split(".")
        drv = (int(p[0]), int(p[1]) if len(p) > 1 else 0)
    except Exception as e:
        print(f"⚠️  Lecture driver impossible : {e}"); return
    ok = drv >= NVVFX_MIN_DRIVER
    print("=" * 60)
    print(f"{'✅ POD COMPATIBLE' if ok else '❌ POD INCOMPATIBLE'} — nvvfx / Nvidia RTX nodes")
    print(f"   GPU    : {gpu}")
    print(f"   Driver : {drv_str}   (requis ≥ 570.26 / CUDA 12.8)")
    if not ok:
        print("   ⛔ Les nodes Nvidia_RTX (NvVFX) ne marcheront PAS sur ce pod.")
        print("   → Terminate + relance avec CUDA 12.8 dans les filtres.")
    print("=" * 60)

check_nvvfx_compat()

# 🚀 DÉMARRAGE V21.30 - ULTIMATE COMMANDER
import os, subprocess, sys, re, json, hashlib, time

# --- CONFIGURATION ---
COMFY_DIR = "/workspace/ComfyUI"
VENV_DIR = "/workspace/my_env"
PYTHON_EXEC = f"{VENV_DIR}/bin/python"
SENTINEL = f"{VENV_DIR}/.install_done_v2129"
LLAMA_SENTINEL = f"{VENV_DIR}/.llama_jame_v0333"
HOTFIX_HASH_FILE = f"{VENV_DIR}/.hotfix_pkgs_hash"
LLAMA_CPP_WHL = "https://github.com/JamePeng/llama-cpp-python/releases/download/v0.3.33-cu124-Basic-linux-20260315/llama_cpp_python-0.3.33%2Bcu124.basic-cp311-cp311-linux_x86_64.whl"

PINNED_VERSIONS = {
    "nvidia-cusparselt-cu12":   "0.7.1",
    "nvidia-nvshmem-cu12":      "3.4.5",
    "nvidia-cuda-cupti-cu12":   "12.8.90",
    "nvidia-cublas-cu12":       "12.8.4.1",
    "nvidia-cuda-runtime-cu12": "12.8.90",
    "nvidia-cudnn-cu12":        "9.10.2.21",
    "nvidia-cufft-cu12":        "11.3.3.83",
    "nvidia-curand-cu12":       "10.3.9.90",
    "nvidia-cusolver-cu12":     "11.7.3.90",
    "nvidia-cusparse-cu12":     "12.5.8.93",
    "nvidia-nccl-cu12":         "2.27.5",
    "nvidia-nvjitlink-cu12":    "12.8.93",
    "numpy":                    "2.3.2",
}

os.environ["PIP_ROOT_USER_ACTION"] = "ignore"
os.environ["PYTHONNOUSERSITE"] = "1"
os.environ["DEBIAN_FRONTEND"] = "noninteractive"
os.environ["MPLBACKEND"] = "Agg"

print("💎 DÉMARRAGE V21.30 - ULTIMATE COMMANDER")
boot_start = time.time()

# ==================================================================================
# ÉTAPE 0 — PIP.CONF + CONSTRAINTS + MANAGER SECURITY
# ==================================================================================
constraints_content = "\n".join(f"{k}=={v}" for k, v in PINNED_VERSIONS.items()) + "\n"
pip_conf_content = f"[global]\nconstraint = {VENV_DIR}/constraints.txt\n"
needs_write = False
if not os.path.exists(f"{VENV_DIR}/constraints.txt") or \
        open(f"{VENV_DIR}/constraints.txt").read() != constraints_content:
    needs_write = True
if not os.path.exists(f"{VENV_DIR}/pip.conf") or \
        open(f"{VENV_DIR}/pip.conf").read() != pip_conf_content:
    needs_write = True
if needs_write:
    os.makedirs(VENV_DIR, exist_ok=True)
    with open(f"{VENV_DIR}/constraints.txt", "w") as f: f.write(constraints_content)
    with open(f"{VENV_DIR}/pip.conf", "w") as f: f.write(pip_conf_content)
    print("✅ pip.conf + constraints.txt mis à jour.")
else:
    print("✅ pip.conf + constraints.txt déjà OK.")

manager_cfg_dir = f"{COMFY_DIR}/user/__manager"
manager_cfg = f"{manager_cfg_dir}/config.ini"
os.makedirs(manager_cfg_dir, exist_ok=True)
if not os.path.exists(manager_cfg) or "security_level = normal" not in open(manager_cfg).read():
    with open(manager_cfg, "w") as f: f.write("[manager]\nsecurity_level = normal\n")
    print("✅ Manager security_level = normal.")

# ==================================================================================
# CACHE GLOBAL & UTILS
# ==================================================================================
_INSTALLED_PKGS_CACHE = None

def get_installed_packages_bulk():
    global _INSTALLED_PKGS_CACHE
    if _INSTALLED_PKGS_CACHE is not None: return _INSTALLED_PKGS_CACHE
    if not os.path.exists(PYTHON_EXEC): return {}
    result = subprocess.run([PYTHON_EXEC, "-m", "pip", "list", "--format=json"],
        capture_output=True, text=True)
    try:
        packages = json.loads(result.stdout)
        _INSTALLED_PKGS_CACHE = {pkg["name"].lower().replace("-", "_"): pkg["version"]
                                  for pkg in packages}
    except: _INSTALLED_PKGS_CACHE = {}
    return _INSTALLED_PKGS_CACHE

def invalidate_pkg_cache():
    global _INSTALLED_PKGS_CACHE
    _INSTALLED_PKGS_CACHE = {}

def run_pip_visible(args_list, no_isolation=False):
    base_cmd = [PYTHON_EXEC, "-m", "pip", "install", "--no-cache-dir", "--prefer-binary"]
    if no_isolation: base_cmd.append("--no-build-isolation")
    subprocess.check_call(base_cmd + args_list)
    invalidate_pkg_cache()

def pip_install_if_missing(packages):
    installed_cache = get_installed_packages_bulk()
    to_install = [pkg for pkg in packages
        if re.split(r'[>=<!;\[]', pkg)[0].strip().lower().replace("-", "_")
        not in installed_cache]
    if to_install:
        print(f"    -> 📦 Install/check ({len(to_install)}) : {', '.join(to_install[:5])}")
        run_pip_visible(to_install)
    else: print("    -> ✅ Librairies Python OK.")

def git_pip_install_if_missing(pkg_name, git_url):
    installed_cache = get_installed_packages_bulk()
    if pkg_name.lower().replace("-", "_") not in installed_cache:
        print(f"    -> 📦 Installation {pkg_name} depuis git...")
        run_pip_visible([git_url])
    else:
        print(f"    -> ✅ {pkg_name} OK.")

# ==================================================================================
# INSTALLATION & PATCHING (SENTINEL CHECK)
# ==================================================================================
if os.path.exists(SENTINEL):
    print("\n⚡ Installation déjà effectuée (v21.29) — skip sections 1 à 4.")
else:
    print("\n🔧 [1/4] Préparation système...")
    sys_pkgs = ["ncdu", "ffmpeg", "git", "cmake", "aria2", "libsndfile1-dev", "build-essential"]
    subprocess.run(["apt-get", "update"], stdout=subprocess.DEVNULL)
    subprocess.run(["apt-get", "install", "-y", "--no-upgrade"] + sys_pkgs,
        stdout=subprocess.DEVNULL)

    print("\n📦 [2/4] Installation de ComfyUI-NAG...")
    nag_path = f"{COMFY_DIR}/custom_nodes/ComfyUI-NAG"
    if not os.path.exists(nag_path):
        subprocess.run(["git", "clone",
            "https://github.com/ChenDarYen/ComfyUI-NAG.git", nag_path])

    if not os.path.exists(f"{VENV_DIR}/bin/python"):
        subprocess.check_call([sys.executable, "-m", "venv", VENV_DIR])

    print("\n🐍 [3/4] Librairies Python...")
    run_pip_visible(["--upgrade", "pip", "wheel", "setuptools"])
    print("    -> 📦 Installation llama-cpp-python JamePeng...")
    run_pip_visible(["--force-reinstall", LLAMA_CPP_WHL])

    pypi_packages = [
        # Core ComfyUI
        "einops", "wand", "accelerate", "scipy", "diffusers", "omegaconf",
        "transformers>=4.48.0", "safetensors", "huggingface-hub",
        # Vision / ML
        "insightface==0.7.3", "onnxruntime-gpu", "onnx",
        "opencv-contrib-python", "opencv-python==4.10.0.84",
        "scikit-image", "scikit-learn", "albumentations",
        "timm", "facexlib", "facenet-pytorch", "piexif", "pilgram",
        # Audio / Video
        "imageio", "imageio-ffmpeg",
        # NLP
        "sentencepiece", "ftfy", "regex", "tokenizers",
        # Utils
        "dill", "joblib", "tqdm", "psutil", "pyyaml", "toml",
        "protobuf", "rich", "typer", "simpleeval", "gitpython",
        "pycocotools", "pywavelets", "numba", "duckdb",
        "rotary-embedding-torch", "iopath", "fairscale",
        "colour-science", "rembg", "pixeloe",
        "bitsandbytes", "peft", "ultralytics",
        "segment-anything", "pygithub",
        # Google / Anthropic
        "google-generativeai", "google-genai", "anthropic",
        # Autres
        "vtracer", "xformers", "sageattention",
    ]
    pip_install_if_missing(pypi_packages)
    git_pip_install_if_missing("sam2", "git+https://github.com/facebookresearch/sam2")

    print("\n🩹 [4/4] Patches de compatibilité...")
    mm_path = f"{COMFY_DIR}/comfy/model_management.py"
    if os.path.exists(mm_path):
        with open(mm_path, 'r') as f: content = f.read()
        if "cu130" in content:
            with open(mm_path, 'w') as f: f.write(content.replace("cu130", "cu128"))
            print("    -> ✅ Patch cu128 appliqué.")

    open(SENTINEL, 'w').close()

# ==================================================================================
# MAINTENANCE & HOTFIXES — VERSION OPTIMISÉE V21.30
# ==================================================================================
print("\n🛠️ [HOTFIX] Application des correctifs...")

# --- 1. Packages pinned — skip si hash identique ---
installed = get_installed_packages_bulk()
current_versions = {k.lower().replace("-", "_"): installed.get(k.lower().replace("-", "_"), "MISSING")
                    for k in PINNED_VERSIONS}
current_hash = hashlib.md5(str(sorted(current_versions.items())).encode()).hexdigest()
saved_hash = open(HOTFIX_HASH_FILE).read().strip() if os.path.exists(HOTFIX_HASH_FILE) else ""

if current_hash != saved_hash:
    to_fix = [f"{k}=={v}" for k, v in PINNED_VERSIONS.items()
              if installed.get(k.lower().replace("-", "_")) != v]
    if to_fix:
        print(f"    -> 🔧 Correction {len(to_fix)} packages pinned incorrects...")
        run_pip_visible(["--force-reinstall"] + to_fix)
        invalidate_pkg_cache()
        # Recalcule le hash après correction
        installed = get_installed_packages_bulk()
        current_versions = {k.lower().replace("-", "_"): installed.get(k.lower().replace("-", "_"), "MISSING")
                            for k in PINNED_VERSIONS}
        current_hash = hashlib.md5(str(sorted(current_versions.items())).encode()).hexdigest()
        print("    -> ✅ Packages pinned corrigés.")
    else:
        print("    -> ✅ Packages pinned OK.")
    with open(HOTFIX_HASH_FILE, "w") as f: f.write(current_hash)
else:
    print("    -> ⚡ Packages pinned OK (hash identique — skip pip list).")

# --- 2. Fix NAG model.py — seulement si pas encore patché ---
nag_model_py = f"{COMFY_DIR}/custom_nodes/ComfyUI-NAG/chroma/model.py"
if os.path.exists(nag_model_py):
    with open(nag_model_py, 'r') as f: content = f.read()
    old_nag_import = """from comfy.ldm.chroma.layers import (
    DoubleStreamBlock,
    SingleStreamBlock,
)
from comfy.ldm.chroma.model import Chroma"""
    if old_nag_import in content:
        # Besoin de patcher : git checkout d'abord pour repartir propre
        subprocess.run(["git", "checkout", "chroma/model.py"],
            cwd=f"{COMFY_DIR}/custom_nodes/ComfyUI-NAG",
            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        with open(nag_model_py, 'r') as f: content = f.read()
        new_nag_import = """try:
    from comfy.ldm.chroma.layers import DoubleStreamBlock, SingleStreamBlock
    from comfy.ldm.chroma.model import Chroma
except (ImportError, AttributeError):
    from comfy.ldm.flux.layers import DoubleStreamBlock, SingleStreamBlock
    from comfy.ldm.flux.model import Flux as Chroma"""
        with open(nag_model_py, 'w') as f: f.write(content.replace(old_nag_import, new_nag_import))
        print("    -> ✅ NAG model.py patché.")
    else:
        print("    -> ⚡ NAG model.py OK (déjà patché — skip git checkout).")

# --- 3. Fix NAG layers.py — seulement si pas encore patché ---
nag_layers_py = f"{COMFY_DIR}/custom_nodes/ComfyUI-NAG/chroma/layers.py"
if os.path.exists(nag_layers_py):
    with open(nag_layers_py, 'r') as f: content = f.read()
    correct_import = "from comfy.ldm.flux.layers import DoubleStreamBlock, SingleStreamBlock"
    if correct_import not in content:
        new_content = re.sub(r'from comfy\.ldm\.chroma\.layers import.*?(?=import|class|def|$)',
            correct_import + "\n", content, flags=re.DOTALL)
        with open(nag_layers_py, 'w') as f: f.write(new_content)
        print("    -> ✅ NAG layers.py patché.")
    else:
        print("    -> ⚡ NAG layers.py OK (skip).")

# --- 4. ImageMagick 6 ---
if not os.path.exists("/usr/lib/x86_64-linux-gnu/libMagickWand-6.Q16.so.6"):
    print("    -> 📦 Installation ImageMagick 6 (apt)...")
    subprocess.run(["apt-get", "install", "-y", "--no-upgrade",
        "libmagickwand-dev", "imagemagick"],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    print("    -> ✅ ImageMagick 6 installé.")
else:
    print("    -> ⚡ ImageMagick 6 OK (skip).")

# --- 5. llama-cpp-python JamePeng — sentinel fichier (skip import subprocess) ---
if not os.path.exists(LLAMA_SENTINEL):
    result = subprocess.run([PYTHON_EXEC, "-c",
        "from llama_cpp.llama_chat_format import Qwen3VLChatHandler, Qwen25VLChatHandler; print('OK')"],
        capture_output=True, text=True)
    if "OK" not in result.stdout:
        print("    -> 📦 Réinstallation llama-cpp-python JamePeng...")
        run_pip_visible(["--force-reinstall", LLAMA_CPP_WHL])
        print("    -> ✅ llama-cpp-python JamePeng installé.")
    open(LLAMA_SENTINEL, "w").close()
    print("    -> ✅ llama-cpp-python JamePeng vérifié + sentinel créé.")
else:
    print("    -> ⚡ llama-cpp-python JamePeng OK (sentinel présent — skip).")

print(f"\n⏱️  Hotfixes terminés en {time.time() - boot_start:.1f}s")

# ==================================================================================
# LANCEMENT (4090 OPTIMIZED)
# ==================================================================================
nvidia_libs = "/usr/local/cuda/lib64:/usr/lib/x86_64-linux-gnu/nvidia:/usr/lib/x86_64-linux-gnu"
os.environ["LD_LIBRARY_PATH"] = f"{nvidia_libs}:" + os.environ.get("LD_LIBRARY_PATH", "")

print("\n🚀 Lancement ComfyUI (Mode HighVRAM)...")
runpod_id = os.environ.get("RUNPOD_POD_ID")
if runpod_id: print(f"\n🔗 LIEN RUNPOD : https://{runpod_id}-8188.proxy.runpod.net")

run_script = f"""#!/bin/bash
source {VENV_DIR}/bin/activate
export LD_LIBRARY_PATH="{nvidia_libs}:$LD_LIBRARY_PATH"
export PYTHONNOUSERSITE=1
export MPLBACKEND=Agg
cd {COMFY_DIR}
python main.py --listen 0.0.0.0 --port 8188 --enable-cors-header --preview-method auto
"""
with open("run_comfy.sh", "w") as f: f.write(run_script)
os.chmod("run_comfy.sh", 0o755)
get_ipython().system('./run_comfy.sh')

## 4. Arreter ComfyUI
A faire avant un snapshot (cellule 7), ou pour relancer ComfyUI.


In [ ]:
%%bash
if pkill -f "main.py --listen"; then sleep 3; echo "ComfyUI arrete."; else echo "ComfyUI ne tournait pas."; fi


## 5. Zipper les outputs
Le zip apparait dans `/workspace`, dans le panneau de gauche de Jupyter : clic droit > Download.


In [ ]:
%%bash
cd /workspace/ComfyUI/output || exit 1
NAME="outputs_$(date +%Y%m%d-%H%M).zip"
if [ -d /local ]; then
  python3 -m zipfile -c "/local/$NAME" . && ln -sf "/local/$NAME" "/workspace/$NAME"
else
  python3 -m zipfile -c "/workspace/$NAME" .
fi
ls -lhL "/workspace/$NAME"

## 6. Mettre a jour ComfyUI et les custom nodes
Dependances installees dans `my_env`. Les nodes modifies localement sont laisses tels quels. Ensuite : relance ComfyUI (cellule 4 puis 3) et teste tes workflows avant tout snapshot.


In [ ]:
%%bash
export SESSION_ENV=image VENV=my_env
PIP=/workspace/$VENV/bin/pip
cd /workspace/ComfyUI || exit 1

echo "=== ComfyUI ==="
DELETED=$(git ls-files --deleted)
[ -n "$DELETED" ] && git checkout -q -- $DELETED
if [ -n "$(git status --porcelain --untracked-files=no)" ]; then
  echo "!! ComfyUI a des modifications locales, mise a jour sautee :"
  git status --short --untracked-files=no
else
  B=$(git rev-parse --short HEAD)
  git fetch -q origin && git checkout -q master && git pull -q --ff-only origin master
  A=$(git rev-parse --short HEAD)
  echo "ComfyUI : $B -> $A"
  [ "$B" != "$A" ] && $PIP install -q -r requirements.txt
fi

echo ""
echo "=== Custom nodes ==="
for d in custom_nodes/*/; do
  n=$(basename "$d")
  [ -d "$d/.git" ] || { echo "  -- $n : pas un depot git, ignore"; continue; }
  if [ -n "$(git -C "$d" status --porcelain --untracked-files=no)" ]; then
    echo "  !! $n : modifie localement, laisse tel quel"; continue
  fi
  B=$(git -C "$d" rev-parse --short HEAD)
  git -C "$d" pull -q --ff-only 2>/dev/null || { echo "  !! $n : mise a jour impossible"; continue; }
  A=$(git -C "$d" rev-parse --short HEAD)
  if [ "$B" != "$A" ]; then
    echo "  OK $n : $B -> $A"
    [ -f "$d/requirements.txt" ] && $PIP install -q -r "$d/requirements.txt"
  fi
done
echo ""
echo "Termine. Relance ComfyUI (cellule 4 puis cellule 3), puis teste tes workflows."

## 7. Snapshot de l'env (nouvelle base pour les prochains pods)
Uniquement apres avoir teste tes workflows. Arrete ComfyUI avant (cellule 4). L'ancien snapshot reste actif tant que le nouveau n'est pas entierement envoye. Evite d'en faire plusieurs le meme jour (place HF liberee sous 36 h).


In [ ]:
%%bash
export SESSION_ENV=image VENV=my_env
export PATH=/workspace/$VENV/bin:$PATH
pgrep -f "main.py --listen" > /dev/null && { echo "ComfyUI tourne encore : lance d'abord la cellule 4."; exit 1; }
curl -sL -H "Authorization: Bearer $HF_TOKEN" https://huggingface.co/rafatribe/Personnal_Loras/resolve/main/session/snapshot_env.sh | bash


## 8. Garder les modeles ajoutes pendant la session (optionnel)
Met a jour la liste du profil "tout" a partir des modeles presents sur le disque.


In [ ]:
%%bash
export SESSION_ENV=image VENV=my_env
export PATH=/workspace/$VENV/bin:$PATH
LISTE=session_models.txt
cd /workspace/ComfyUI/models || exit 1
find -L . -type f \( -name "*.safetensors" -o -name "*.ckpt" -o -name "*.pt" -o -name "*.pth" -o -name "*.gguf" -o -name "*.bin" \) -size +0 \
  | sed 's#^\./##' | grep -v '^sam3/sam3.pt$' | grep -v '/\.cache/' | sort > /tmp/session_models.txt
echo "$(wc -l < /tmp/session_models.txt) modeles dans la liste $LISTE (profil \"tout\")"
hf upload rafatribe/Personnal_Loras /tmp/session_models.txt "session/$LISTE" --commit-message "Liste modeles $(date +%F)"